# Predicting term-deposit subscription (`y`) after a marketing call
**Question:** can we predict, from information available *before/at the time of the call*, whether a client subscribes?

**Leakage note:** `duration` is only known after the call ends (and a call that ends in a sale is naturally long), so it is **excluded** from all features. A model using it could not be used to decide whom to call. We quantify the inflation it would cause at the end, for illustration only.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
SEED=42
df=pd.read_csv('data.csv'); print(df.shape)
print(df.dtypes); print(df.isna().mean().round(3)); print(df.duplicated().sum(),'duplicate rows')
df['y'].value_counts(normalize=True)

Target is imbalanced (~12% yes), so we use ROC-AUC and PR-AUC (average precision) rather than accuracy. Missing values in `contact`/`poutcome` are treated as their own 'missing' category.

In [ ]:
y=(df['y']=='yes').astype(int)
X=df.drop(columns=['y','duration'])   # duration dropped: post-call information (leakage)
cat=X.select_dtypes('object').columns.tolist(); num=[c for c in X.columns if c not in cat]
print(cat, num)
fig,ax=plt.subplots(1,3,figsize=(15,3.5))
for a,col in zip(ax,['job','month','poutcome']):
    df.fillna({col:'missing'}).groupby(col)['y'].apply(lambda s:(s=='yes').mean()).sort_values().plot.barh(ax=a,title=f'Subscription rate by {col}')
plt.tight_layout(); plt.show()

In [ ]:
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
def prep(scale):
    return ColumnTransformer([
        ('c',Pipeline([('i',SimpleImputer(strategy='constant',fill_value='missing')),('o',OneHotEncoder(handle_unknown='ignore'))]),cat),
        ('n',StandardScaler() if scale else 'passthrough',num)])
models={
 'baseline (prior)':Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logistic regression':Pipeline([('p',prep(True)),('m',LogisticRegression(max_iter=2000))]),
 'gradient boosting':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(learning_rate=0.05,max_iter=300,max_leaf_nodes=15,l2_regularization=1.0,early_stopping=True,random_state=SEED))]),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[]
for n,m in models.items():
    s=cross_val_score(m,Xtr,ytr,cv=cv,scoring='roc_auc'); p=cross_val_score(m,Xtr,ytr,cv=cv,scoring='average_precision')
    rows.append((n,s.mean(),s.std(),p.mean()))
pd.DataFrame(rows,columns=['model','cv_roc_auc','std','cv_pr_auc']).round(3)

Model comparison uses cross-validation on the training set only; the held-out test set is used once, below.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4)); res={}
for n,m in models.items():
    m.fit(Xtr,ytr); pr=m.predict_proba(Xte)[:,1]
    res[n]=(roc_auc_score(yte,pr),average_precision_score(yte,pr))
    if 'baseline' not in n:
        RocCurveDisplay.from_predictions(yte,pr,name=n,ax=ax[0]); PrecisionRecallDisplay.from_predictions(yte,pr,name=n,ax=ax[1])
ax[1].axhline(yte.mean(),ls='--',c='grey'); plt.show()
pd.DataFrame(res,index=['test_roc_auc','test_pr_auc']).T.round(3)

In [ ]:
# Business view: contacting only the top-scored clients (gradient boosting)
pr=models['gradient boosting'].predict_proba(Xte)[:,1]
d=pd.DataFrame({'y':yte.values,'p':pr}).sort_values('p',ascending=False)
for f in [.1,.2,.3,.5]:
    t=d.head(int(len(d)*f)); print(f'top {f:.0%} of clients: conversion {t.y.mean():.1%}, captures {t.y.sum()/d.y.sum():.0%} of all subscribers (base rate {d.y.mean():.1%})')

In [ ]:
from sklearn.inspection import permutation_importance
pi=permutation_importance(models['gradient boosting'],Xte,yte,scoring='roc_auc',n_repeats=5,random_state=SEED)
pd.Series(pi.importances_mean,index=X.columns).sort_values().plot.barh(title='Permutation importance (test ROC-AUC drop)'); plt.show()

## Illustration: what including `duration` would do (do NOT use)

In [ ]:
Xl=df.drop(columns=['y'])
leak=Pipeline([('p',ColumnTransformer([('c',Pipeline([('i',SimpleImputer(strategy='constant',fill_value='missing')),('o',OneHotEncoder(handle_unknown='ignore'))]),cat),('n','passthrough',num+['duration'])])),
               ('m',HistGradientBoostingClassifier(learning_rate=0.05,max_iter=300,max_leaf_nodes=15,random_state=SEED))]).fit(Xl.loc[Xtr.index],ytr)
print('test ROC-AUC with duration (leaky):',round(roc_auc_score(yte,leak.predict_proba(Xl.loc[Xte.index])[:,1]),3))

## Conclusions
Yes, partly. Without `duration`, gradient boosting reaches **test ROC-AUC ≈ 0.80 and PR-AUC ≈ 0.47** (baseline 0.50 / 0.12); logistic regression gets 0.77 / 0.41. Cross-validation agrees (0.797 ± 0.004).

Practical value is in ranking: calling only the top-scored 20% of clients gives ~36% conversion (vs. 11.7% base rate) and captures ~62% of all subscribers. Individual outcomes remain far from certain.

Adding `duration` lifts ROC-AUC to ≈ 0.93, but it is unavailable before the call, so that number is not achievable in practice.

**Caveats:** there is no date/year column, so a random split may be optimistic compared with a real future-time split; `pdays`/`previous`/`poutcome` describe prior campaigns (`poutcome` is ~82% missing), so check they are available for new clients; probabilities are not calibrated; no hyperparameter tuning was done.

*Note: outputs were not saved in this file (kernel could not start in the sandbox); run all cells to populate them.*